1. # Enable GPU & Mount Google Drive

In [ ]:
import torch
from google.colab import drive

# 1. Check GPU status
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Running on CPU. Go to Runtime -> Change runtime type -> Select T4 GPU.")

# 2. Mount Drive
drive.mount('/content/drive')

2. # Clone GitHub Repository & Install Requirements

In [ ]:
import os

# Update with your GitHub username and repository name
GITHUB_USER = "Maina-Mwafrika"
REPO_NAME = "chord-detector-trainer"
REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

%cd {REPO_NAME}

# Pull latest commits if re-running
!git pull origin main

# Install project dependencies
!pip install -r requirements.txt

3. # Define 25-Class PyTorch Model Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ChordCRNN(nn.Module):
    def __init__(self, num_classes=25):
        super().__init__()
        # Input shape: (Batch, 3, 84, TimeFrames) -> [CQT, Delta1, Delta2]
        self.conv_block = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),  # Halve frequency bins (84 -> 42)
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),  # Halve frequency bins (42 -> 21)
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d((2, 1))   # Halve frequency bins (21 -> 10)
        )
        
        # Bidirectional GRU for modeling harmonic progression context
        self.gru = nn.GRU(
            input_size=128 * 10,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )
        
        # Output Linear Projection
        self.fc = nn.Linear(128 * 2, num_classes)

    def forward(self, x):
        # x: (B, C, F, T)
        b, c, f, t = x.shape
        x = self.conv_block(x)  # (B, 128, 10, T)
        
        # Reshape for Sequential Recurrent Input: (B, T, Features)
        x = x.permute(0, 3, 1, 2).contiguous()
        x = x.view(b, t, -1)
        
        gru_out, _ = self.gru(x)  # (B, T, 256)
        logits = self.fc(gru_out)  # (B, T, 255)
        
        return logits

model = ChordCRNN(num_classes=25)
print("Model created successfully. Parameter count:", sum(p.numel() for p in model.parameters()))

4. # Synthetic DataLoader & GPU Training Loop

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Replace synthetic data with actual preprocessed CQT tensors from Drive when ready
# Dummy Batch: 16 sequences, 3 channels, 84 CQT bins, 200 time frames
dummy_x = torch.randn(16, 3, 84, 200)
dummy_y = torch.randint(0, 25, (16, 200))  # Ground truth frame targets

dataset = TensorDataset(dummy_x, dummy_y)
train_loader = DataLoader(dataset, batch_size=4, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

epochs = 5
print(f"Starting training on device: {device}")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        logits = model(batch_x)  # (B, T, 25)
        
        # Flatten time dimension for CrossEntropy Loss
        loss = criterion(logits.view(-1, 25), batch_y.view(-1))
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {avg_loss:.4f}")

5. # ONNX Model Export to Google Drive

In [ ]:
import os

os.makedirs("/content/drive/MyDrive/ChordDetectorModels", exist_ok=True)
onnx_save_path = "/content/drive/MyDrive/ChordDetectorModels/chord_recognizer_24class.onnx"

model.eval()
model.to("cpu")

# Dummy input matching shape: (Batch, Channels, CQT Bins, TimeFrames)
dummy_input = torch.randn(1, 3, 84, 200)

torch.onnx.export(
    model,
    dummy_input,
    onnx_save_path,
    input_names=["cqt_input"],
    output_names=["logits"],
    dynamic_axes={
        "cqt_input": {0: "batch_size", 3: "time_frames"},
        "logits": {0: "batch_size", 1: "time_frames"}
    },
    opset_version=14
)

print(f"ONNX Model saved successfully to Google Drive:\n{onnx_save_path}")